In [17]:
!pip install langchain-groq==1.1.3, langfuse==4.14.5, langchain_openai==1.6.0 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 4.9 MB/s eta 0:00:00


In [2]:
# Parser

# Parser

- Output parsers are responsible for taking the output of an LLM and transforming it to a more suitable format.
- This is very useful when you are using LLMs to generate any form of structured data.
- Very helpful when AI usecase in production
- Latest models are compatiable with output parsers

In [18]:
# from langchain.output_parsers import StructuredOutputParser, ResponseSchema
# from langchain.output_parsers import StructuredOutputParser, ResponseSchema

from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

In [19]:
import os


from google.colab import userdata
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# To create llm object, make sure you have 'GROQ_API_KEY' in your environment variable
# comment above line of code where we are fetching colab userdata.

# llm = ChatGroq(temperature=0.6, model_name="openai/gpt-oss-120b")
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0.3)
response = llm.invoke("hi")

In [20]:
response.content

'Hello! How can I assist you today?'

In [21]:
prompt = "write an email to my boss seeking 2 days leave with subject and body."

response = llm.invoke(prompt).content
print(response)

Subject: Request for Two Days Leave

Dear [Boss's Name],

I hope this message finds you well. I would like to request two days of leave on [start date] and [end date] due to [brief reason, e.g., personal matters/family commitments]. I will ensure that all my current tasks are up to date before my leave and will be available to assist remotely if needed.

Please let me know if you need any additional information. Thank you for your consideration.

Best regards,  
[Your Name]


In [25]:
from pydantic import BaseModel, Field

class EmailParser(BaseModel):
    email: str = Field(description="A Formal mail writen to boss on given subject with proper formatting and grammer.")
    subject: str = Field(description="Subject of the email")

# creating llm object with structured output parser
llm_parser = llm.with_structured_output(EmailParser)

response = llm_parser.invoke(prompt)

In [26]:
print(response.email)

Dear [Boss's Name],

I hope this message finds you well. I would like to request leave for two days, from [start date] to [end date], due to personal reasons. I will ensure that all my current tasks are up to date before my leave and will be available for any urgent matters.

Thank you for considering my request. Please let me know if you need any further information.

Best regards,
[Your Name]


In [27]:
response.subject

'Request for Two Days Leave'

In [28]:
response

EmailParser(email="Dear [Boss's Name],\n\nI hope this message finds you well. I would like to request leave for two days, from [start date] to [end date], due to personal reasons. I will ensure that all my current tasks are up to date before my leave and will be available for any urgent matters.\n\nThank you for considering my request. Please let me know if you need any further information.\n\nBest regards,\n[Your Name]", subject='Request for Two Days Leave')

In [29]:
llm_parser

_ChatModelBinding(bound=ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x783be899c2f0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x783be899c830>, root_client=<openai.OpenAI object at 

In [ ]:
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, client=<groq.resources.chat.completions.Completions object at 0x7c1162a0b4d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c11627c02f0>, model_name='qwen/qwen3.8-27b', temperature=0.6, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [30]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langfuse import Langfuse, get_client
from langfuse.langchain import CallbackHandler
from langchain_core.prompts import ChatPromptTemplate

# Initialize Langfuse client with constructor arguments
langfuse = Langfuse(
    public_key=userdata.get("LANGFUSE_PUBLIC_KEY"),
    secret_key=userdata.get("LANGFUSE_SECRET_KEY"),
    host="https://us.cloud.langfuse.com",
)

# # Get the configured client instance
# langfuse = get_client()

print(langfuse.auth_check())

# Initialize the Langfuse handler
langfuse_handler = CallbackHandler(public_key=userdata.get("LANGFUSE_PUBLIC_KEY"))


parser = PydanticOutputParser(pydantic_object = EmailParser)


prompt = """write an email to my boss seeking 2 days leave with subject and body.

## Reason for leave:

Reason for applying leave is: '{reason}'
---
## output instructions:

{format_instructions}
"""

chat_prompt = ChatPromptTemplate.from_template(
    prompt,
    partial_variables={
        "format_instructions": parser.get_format_instructions()
    },
)


chain = chat_prompt | llm | parser


response = chain.invoke({'reason': 'personal reason'}, config={"callbacks": [langfuse_handler]})

True


In [31]:
print(response.email)

Dear [Boss's Name],

I hope this message finds you well. I am writing to formally request leave for two days due to personal reasons. I would appreciate your approval for leave on [start date] and [end date].

Please let me know if you need any further information or if there are any urgent tasks I should address before my leave.

Thank you for your understanding.

Best regards,
[Your Name]


In [32]:
response.subject

'Request for Two Days Leave'